# Setup

In [1]:
import os
import sys
parent_dir = os.path.abspath(os.path.join(os.path.dirname("inpire.py"), '..'))
sys.path.append(parent_dir)

In [79]:
from inspire import InspireClassifier

import logging
import warnings
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from imblearn.over_sampling import BorderlineSMOTE
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo 
import kagglehub
from kagglehub import KaggleDatasetAdapter
from tqdm.notebook import tqdm
import json

np.random.seed(42)
logging.basicConfig(level=logging.INFO)
warnings.simplefilter("ignore")

In [3]:
def generate_training_raport(model, X_test, y_test):
    y_pred = model.predict(X_test, level=logging.CRITICAL)
    print(classification_report(y_test, y_pred), "\n")

    n = int(len(model.history[3:]) / 2)
    print("_________Training report_________\n")
    print(f"Removed noise: {sum(model.history[0]['removed_mask'])}")
    if "border_mask" in model.history[1]:
        oversampling = True
        print(f"Border samples: {sum(model.history[1]['border_mask'])}")
    else:
        oversampling = False

    if "folds" in model.history[2]:
        print(f"Number of folds: {len(model.history[2]['folds'])}")

    print()

    for i in range(n):
        print(f"Batch {i+1}")
        if i > 0:
            changed_preds = sum(
                model.history[2 * i + 3]["preds"]
                != model.history[2 * (i - 1) + 3]["preds"]
            )
            print(f"Changed predictions: {changed_preds}")
        if oversampling:
            print(f"Wrong predictions: {sum(model.history[2*i+4]['misclass_mask'])}")
            print(
                f"Low confidence in predictions: {sum(model.history[2*i+4]['condidence_mask'])}"
            )
            print(f"Val BP samples: {sum(model.history[2*i+4]['val_bp_mask'])}")
            print(f"BP samples: {sum(model.history[2*i+4]['bp_mask'])}")
            print(
                f"Oversampling indces: {len(model.history[2*i+4]['oversampling_indeces'])}"
            )
            print(f"X_synth size: {len(model.history[2*i+4]['X_synth'])}")
        print()

    changed_preds = sum(
        model.history[3]["preds"] != model.history[2 * (n - 1) + 3]["preds"]
    )
    print(f"Changed preds (1 -> {n} batch): {changed_preds}")

In [76]:
def test_model(
    model_kwargs,
    train_kwargs,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    minority_class,
    iterations=10,
):
    recalls = {}
    precisions = {}
    fscores = {}
    accuracies = {}
    for sampling_ratio in [0.1, 0.5, 1]:
        recalls_i = []
        precisions_i = []
        fscores_i = []
        accuracies_i = []
        for i in tqdm(range(iterations)):
            model_i = InspireClassifier(**model_kwargs)
            model_i.fit(
                X_train,
                y_train,
                X_val,
                y_val,
                **train_kwargs,
                oversampling_ratio=sampling_ratio,
            )

            y_pred = model_i.predict(
                X_test,
                level=logging.CRITICAL,
            )
            report = classification_report(y_test, y_pred, output_dict=True)

            recalls_i.append(report[f"{minority_class}"]["recall"])
            precisions_i.append(report[f"{minority_class}"]["precision"])
            fscores_i.append(report[f"{minority_class}"]["f1-score"])
            accuracies_i.append(report["accuracy"])

        recalls[sampling_ratio] = recalls_i
        precisions[sampling_ratio] = precisions_i
        fscores[sampling_ratio] = fscores_i
        accuracies[sampling_ratio] = accuracies_i
    return {"recalls": recalls, "precisions": precisions, "fscores": fscores, "accuracies": accuracies}

In [35]:
def test_rfc(X_train, y_train, X_test, y_test, minority_class, rfc_kwargs, sampling_ratios):
    recalls = {}
    precisions = {}
    fscores = {}
    accuracies = {}

    for sampling_ratio in sampling_ratios:
        recalls_i = []
        precisions_i = []
        fscores_i = []
        accuracies_i = []
        for i in tqdm(range(10)):
            rfc_i = RandomForestClassifier(**rfc_kwargs)

            smote = BorderlineSMOTE(
                kind="borderline-1", random_state=42, sampling_strategy=sampling_ratio
            )

            X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

            rfc_i.fit(X_resampled, y_resampled)
    
            y_pred = rfc_i.predict(X_test)
            report = classification_report(y_test, y_pred, output_dict=True)

            recalls_i.append(report[f"{minority_class}"]["recall"])
            precisions_i.append(report[f"{minority_class}"]["precision"])
            fscores_i.append(report[f"{minority_class}"]["f1-score"])
            accuracies_i.append(report["accuracy"])

        recalls[sampling_ratio] = recalls_i
        precisions[sampling_ratio] = precisions_i
        fscores[sampling_ratio] = fscores_i
        accuracies[sampling_ratio] = accuracies_i
    return {
        "recalls": recalls,
        "precisions": precisions,
        "fscores": fscores,
        "accuracies": accuracies,
    }

In [6]:
scenarios = ["no_bagging", "all_features", "no_smote"]

# Polish companies bankruptcy

## Data

In [7]:
polish_companies_bankruptcy = fetch_ucirepo(id=365) 
  
X = polish_companies_bankruptcy.data.features
y = polish_companies_bankruptcy.data.targets

In [8]:
df = pd.DataFrame(X)
df["target"] = y

In [9]:
df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19967 entries, 0 to 43400
Data columns (total 66 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   year    19967 non-null  int64  
 1   A1      19967 non-null  float64
 2   A2      19967 non-null  float64
 3   A3      19967 non-null  float64
 4   A4      19967 non-null  float64
 5   A5      19967 non-null  float64
 6   A6      19967 non-null  float64
 7   A7      19967 non-null  float64
 8   A8      19967 non-null  float64
 9   A9      19967 non-null  float64
 10  A10     19967 non-null  float64
 11  A11     19967 non-null  float64
 12  A12     19967 non-null  float64
 13  A13     19967 non-null  float64
 14  A14     19967 non-null  float64
 15  A15     19967 non-null  float64
 16  A16     19967 non-null  float64
 17  A17     19967 non-null  float64
 18  A18     19967 non-null  float64
 19  A19     19967 non-null  float64
 20  A20     19967 non-null  float64
 21  A21     19967 non-null  float64
 22  A22

In [10]:
y = df["target"].to_numpy()
X = df.drop("target", axis=1).to_numpy()

In [11]:
scaler = StandardScaler() 
X = scaler.fit_transform(X)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [13]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 14648, np.int64(1): 327})
test disbalance: Counter({np.int64(0): 2436, np.int64(1): 60})
val disbalance: Counter({np.int64(0): 2451, np.int64(1): 45})


In [14]:
minority_class = 1

## TESTS

In [15]:
model_kwargs = {
    "base_estimator": DecisionTreeClassifier(
        max_depth=20,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
}

fit_kwargs = {
    "save_history_": False,
    "cleanup_": True,
    "level": logging.CRITICAL,
    "bp_kwargs":{"neighbours": 2},
    "br_kwargs":{"br_treshold": 0.7},
}

In [16]:
polish_comp_results = {}

for scenario in scenarios:
    if scenario == "all_features":
        model_kwargs["n_estimators"] = 50
        fit_kwargs["bagging_"] = True
        fit_kwargs["perform_oversampling_"] = True
        fit_kwargs["step_size"] = 2
        
    elif scenario == "no_bagging":
        model_kwargs["n_estimators"] = 1
        fit_kwargs["bagging_"] = False
        fit_kwargs["perform_oversampling_"] = True
        fit_kwargs["step_size"] = 1
        
    elif scenario == "no_smote":
        model_kwargs["n_estimators"] = 50
        fit_kwargs["bagging_"] = True
        fit_kwargs["perform_oversampling_"] = False
        fit_kwargs["step_size"] = 2
        
    results = test_model(
        model_kwargs,
        fit_kwargs,
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        minority_class,
    )
    
    polish_comp_results[scenario] = results

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [19]:
polish_comp_results["rfc"] = test_rfc(
    X_train,
    y_train,
    X_test,
    y_test,
    minority_class,
    rfc_kwargs={
        "n_estimators": 50,
        "max_depth": 20,
        "min_samples_split": 5,
        "min_samples_leaf": 3,
        "max_features": 0.8,
        "random_state": 42,
    },
)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [20]:
polish_comp_results

{'no_bagging': {'recalls': {0.1: [0.3,
    0.3,
    0.3,
    0.3,
    0.3,
    0.3,
    0.3,
    0.3,
    0.3,
    0.3],
   0.5: [0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3],
   1: [0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3]},
  'precisions': {0.1: [0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102],
   0.5: [0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102],
   1: [0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102,
    0.3673469387755102]},
  'fscore

In [81]:
ada_recalls = {}
ada_precisions = {}
ada_fscores = {}
ada_accuracies = {}

for lr in [0.1, 0.5, 1]:
    recalls_i = []
    precisions_i = []
    fscores_i = []
    accuracies_i = []
    for i in tqdm(range(10)):
        adaboost_i = AdaBoostClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=20,
                min_samples_split=5,
                min_samples_leaf=3,
                max_features=0.8,
                random_state=42,
            ),
            n_estimators=50,
            learning_rate=lr,
            random_state=42,
        )
        
        adaboost_i.fit(X_train, y_train)
        y_pred = adaboost_i.predict(X_test)
        report = classification_report(y_test, y_pred, output_dict=True)
        recalls_i.append(report[f"{minority_class}"]["recall"])
        precisions_i.append(report[f"{minority_class}"]["precision"])
        fscores_i.append(report[f"{minority_class}"]["f1-score"])
        accuracies_i.append(report["accuracy"])
        
    ada_recalls[lr] = recalls_i
    ada_precisions[lr] = precisions_i
    ada_fscores[lr] = fscores_i
    ada_accuracies[lr] = accuracies_i

  0%|          | 0/10 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [21]:
with open("results/polish_companies_results_2.json", "w") as f:
    json.dump(polish_comp_results, f, indent=4)

# Breast Cancer Wisconsin

## Data

In [22]:
breast_cancer_wisconsin_diagnostic = fetch_ucirepo(id=17) 
  
X = breast_cancer_wisconsin_diagnostic.data.features 
y = breast_cancer_wisconsin_diagnostic.data.targets

In [23]:
df = pd.DataFrame(X)
df["target"] = y

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   radius1             569 non-null    float64
 1   texture1            569 non-null    float64
 2   perimeter1          569 non-null    float64
 3   area1               569 non-null    float64
 4   smoothness1         569 non-null    float64
 5   compactness1        569 non-null    float64
 6   concavity1          569 non-null    float64
 7   concave_points1     569 non-null    float64
 8   symmetry1           569 non-null    float64
 9   fractal_dimension1  569 non-null    float64
 10  radius2             569 non-null    float64
 11  texture2            569 non-null    float64
 12  perimeter2          569 non-null    float64
 13  area2               569 non-null    float64
 14  smoothness2         569 non-null    float64
 15  compactness2        569 non-null    float64
 16  concavit

In [25]:
X = df.drop("target", axis=1).to_numpy()
y = df["target"].to_numpy()

In [26]:
Counter(y)

Counter({'B': 357, 'M': 212})

In [27]:
y = (y == 'M').astype(int)

In [28]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [30]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 268, np.int64(1): 158})
test disbalance: Counter({np.int64(0): 42, np.int64(1): 29})
val disbalance: Counter({np.int64(0): 47, np.int64(1): 25})


In [31]:
minority_class = 1

## TESTS

In [32]:
model_kwargs = {
    "base_estimator": DecisionTreeClassifier(
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
}

fit_kwargs = {
    "save_history_": False,
    "cleanup_": True,
    "level": logging.CRITICAL,
    "bp_kwargs":{"neighbours": 2},
    "br_kwargs":{"br_treshold": 0.7},
}

In [33]:
breast_cancer_results = {}

for scenario in scenarios:
    if scenario == "all_features":
        model_kwargs["n_estimators"] = 50
        fit_kwargs["bagging_"] = True
        fit_kwargs["perform_oversampling_"] = True
        fit_kwargs["step_size"] = 2
        
    elif scenario == "no_bagging":
        model_kwargs["n_estimators"] = 1
        fit_kwargs["bagging_"] = False
        fit_kwargs["perform_oversampling_"] = True
        fit_kwargs["step_size"] = 1
        
    elif scenario == "no_smote":
        model_kwargs["n_estimators"] = 50
        fit_kwargs["bagging_"] = True
        fit_kwargs["perform_oversampling_"] = False
        fit_kwargs["step_size"] = 2
        
    results = test_model(
        model_kwargs,
        fit_kwargs,
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        minority_class,
    )
    
    breast_cancer_results[scenario] = results

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [36]:
breast_cancer_results["rfc"] = test_rfc(
    X_train,
    y_train,
    X_test,
    y_test,
    minority_class,
    rfc_kwargs={
        "n_estimators": 50,
        "max_depth": 10,
        "min_samples_split": 5,
        "min_samples_leaf": 3,
        "max_features": 0.8,
        "random_state": 42,
    },
    sampling_ratios=[1]
)

  0%|          | 0/10 [00:00<?, ?it/s]

In [37]:
with open("results/breast_cancer_results_2.json", "w") as f:
    json.dump(breast_cancer_results, f, indent=4)

# Ionosphere

## Data

In [38]:
ionosphere = fetch_ucirepo(id=52) 
  
X = ionosphere.data.features 
y = ionosphere.data.targets 

In [39]:
df = pd.DataFrame(X)
df["target"] = y

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351 entries, 0 to 350
Data columns (total 35 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Attribute1   351 non-null    int64  
 1   Attribute2   351 non-null    int64  
 2   Attribute3   351 non-null    float64
 3   Attribute4   351 non-null    float64
 4   Attribute5   351 non-null    float64
 5   Attribute6   351 non-null    float64
 6   Attribute7   351 non-null    float64
 7   Attribute8   351 non-null    float64
 8   Attribute9   351 non-null    float64
 9   Attribute10  351 non-null    float64
 10  Attribute11  351 non-null    float64
 11  Attribute12  351 non-null    float64
 12  Attribute13  351 non-null    float64
 13  Attribute14  351 non-null    float64
 14  Attribute15  351 non-null    float64
 15  Attribute16  351 non-null    float64
 16  Attribute17  351 non-null    float64
 17  Attribute18  351 non-null    float64
 18  Attribute19  351 non-null    float64
 19  Attribut

In [41]:
X = df.drop("target", axis=1).to_numpy()
y = df["target"].to_numpy()

In [42]:
Counter(y)

Counter({'g': 225, 'b': 126})

In [43]:
y = (y == 'b').astype(int)

In [44]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [46]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 169, np.int64(1): 94})
test disbalance: Counter({np.int64(0): 25, np.int64(1): 19})
val disbalance: Counter({np.int64(0): 31, np.int64(1): 13})


In [47]:
minority_class = 1

## TESTS

In [48]:
model_kwargs = {
    "base_estimator": DecisionTreeClassifier(
        max_depth=14,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
}

fit_kwargs = {
    "save_history_": False,
    "cleanup_": True,
    "level": logging.CRITICAL,
    "bp_kwargs":{"neighbours": 2},
    "br_kwargs":{"br_treshold": 0.7},
}

In [49]:
ionosphere_results = {}

for scenario in scenarios:
    if scenario == "all_features":
        model_kwargs["n_estimators"] = 50
        fit_kwargs["bagging_"] = True
        fit_kwargs["perform_oversampling_"] = True
        fit_kwargs["step_size"] = 2
        
    elif scenario == "no_bagging":
        model_kwargs["n_estimators"] = 1
        fit_kwargs["bagging_"] = False
        fit_kwargs["perform_oversampling_"] = True
        fit_kwargs["step_size"] = 1
        
    elif scenario == "no_smote":
        model_kwargs["n_estimators"] = 50
        fit_kwargs["bagging_"] = True
        fit_kwargs["perform_oversampling_"] = False
        fit_kwargs["step_size"] = 2
        
    results = test_model(
        model_kwargs,
        fit_kwargs,
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        minority_class,
    )
    
    ionosphere_results[scenario] = results

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [50]:
ionosphere_results["rfc"] = test_rfc(
    X_train,
    y_train,
    X_test,
    y_test,
    minority_class,
    rfc_kwargs={
        "n_estimators": 50,
        "max_depth": 14,
        "min_samples_split": 5,
        "min_samples_leaf": 3,
        "max_features": 0.8,
        "random_state": 42,
    },
    sampling_ratios=[1]
)

  0%|          | 0/10 [00:00<?, ?it/s]

In [51]:
with open("results/ionosphere_results_2.json", "w") as f:
    json.dump(ionosphere_results, f, indent=4)

# Pima Indians Diabetes

## Data

In [52]:
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "uciml/pima-indians-diabetes-database",
  "diabetes.csv",
  )

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [54]:
X = df.drop("Outcome", axis=1).to_numpy()
y = df["Outcome"].to_numpy()

In [55]:
Counter(y)

Counter({np.int64(0): 500, np.int64(1): 268})

In [56]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)
X_test, X_val, y_test, y_val = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42, shuffle=True
)

In [58]:
print("train disbalance:", Counter(y_train))
print("test disbalance:", Counter(y_test))
print("val disbalance:", Counter(y_val))

train disbalance: Counter({np.int64(0): 377, np.int64(1): 199})
test disbalance: Counter({np.int64(0): 64, np.int64(1): 32})
val disbalance: Counter({np.int64(0): 59, np.int64(1): 37})


In [59]:
minority_class = 1

## TESTS

In [60]:
model_kwargs = {
    "base_estimator": DecisionTreeClassifier(
        max_depth=4,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
    ),
}

fit_kwargs = {
    "save_history_": False,
    "cleanup_": True,
    "level": logging.CRITICAL,
    "bp_kwargs":{"neighbours": 2},
    "br_kwargs":{"br_treshold": 0.7},
}

In [61]:
diabetes_results = {}

for scenario in scenarios:
    if scenario == "all_features":
        model_kwargs["n_estimators"] = 50
        fit_kwargs["bagging_"] = True
        fit_kwargs["perform_oversampling_"] = True
        fit_kwargs["step_size"] = 2
        
    elif scenario == "no_bagging":
        model_kwargs["n_estimators"] = 1
        fit_kwargs["bagging_"] = False
        fit_kwargs["perform_oversampling_"] = True
        fit_kwargs["step_size"] = 1
        
    elif scenario == "no_smote":
        model_kwargs["n_estimators"] = 50
        fit_kwargs["bagging_"] = True
        fit_kwargs["perform_oversampling_"] = False
        fit_kwargs["step_size"] = 2
        
    results = test_model(
        model_kwargs,
        fit_kwargs,
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test,
        minority_class,
    )
    
    diabetes_results[scenario] = results

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [62]:
diabetes_results["rfc"] = test_rfc(
    X_train,
    y_train,
    X_test,
    y_test,
    minority_class,
    rfc_kwargs={
        "n_estimators": 50,
        "max_depth": 4,
        "min_samples_split": 5,
        "min_samples_leaf": 3,
        "max_features": 0.8,
        "random_state": 42,
    },
    sampling_ratios=[1],
)

  0%|          | 0/10 [00:00<?, ?it/s]

In [63]:
with open("results/diabetes_results_2.json", "w") as f:
    json.dump(diabetes_results, f, indent=4)